In [1]:
!pip install pyspark
from pyspark import SparkConf, SparkContext
conf = SparkConf().setAppName("Prova_esame")
sc = SparkContext(conf=conf)

In [2]:
outputPath1="./output1/"
WatchedRecorderRDD=sc.textFile("./data/StudentsWatchedRecordedLectures.txt")
RecordedLecturesRDD=sc.textFile("./data/RecordedLectures.txt")

#Task 1

In [68]:
cleanedLecturesRDD=RecordedLecturesRDD.map(lambda x: (x.split(",")[0],x.split(",")[1])).cache()

cleanedWatchedRDD=WatchedRecorderRDD.map(lambda x: (x.split(",")[2],(x.split(",")[0],x.split(",")[1].split("/")[0]))) \
    .filter(lambda x: int(x[1][1])>=2015 and int(x[0][1])<=2020).distinct()

joinedRDD=cleanedWatchedRDD.join(cleanedLecturesRDD).map(lambda x: ((x[1][0][0],x[1][0][1]),x[1][1])).filter(lambda x: x[1]=='CID10') \
    .map(lambda x: (x[0],1)).reduceByKey(lambda a,b: a+b)

# Task 2

In [114]:
outputPath2="./output2/"

In [115]:
newCleanedWatchedRDD=WatchedRecorderRDD.map(lambda x: (x.split(",")[2],x.split(",")[1].split("/")[0])) \
    .filter(lambda x: x[1]=='2023' or x[1]=='2024')

newJoinedRDD=newCleanedWatchedRDD.join(cleanedLecturesRDD).map(lambda x: ((x[1][1],x[1][0]),1)) \
    .reduceByKey(lambda a,b: a+b).cache()

min2023 = newJoinedRDD.filter(lambda x: x[0][1] == '2023').values().min()
max2024 = newJoinedRDD.filter(lambda x: x[0][1] == '2024').values().max()

transformJoinedRDD=newJoinedRDD.map(lambda x: (x[0][0],(x[0][1],x[1]))).groupByKey()

def pickValues(row):
  key, vals=row
  num23=0
  num24=0
  for val in vals:
    if val[0]=='2023':
      num23=val[1]
    elif val[0]=='2024':
      num24=val[1]
  if num23==min2023 and num24==max2024:
    return (key,(num23,num24))

finalRDD=transformJoinedRDD.map(pickValues).filter(lambda x: x is not None)
numELem=finalRDD.count()
if numELem!=0:
  finalRDD.saveAsTextFile(outputPath2)

